# 04 — Ghép fact và huấn luyện Spark LR/GBT

### 1. Dựng fact từ Silver và dimension

Chạy sau notebook 03. `steps.build_fact()` ghép mỗi đơn Silver với phiên bản
shipper theo ngày nhận và các khóa vùng, AOI, ngày nhận, ngày giao, giờ nhận.
Hàm ghi `fact_delivery_task_gold` và đối soát khóa với Silver.

In [ ]:
import sys

sys.path.insert(0, "/home/jovyan/pipeline")
import steps

spark = steps.get_spark("Delivery - Fact and ML Gold")

In [ ]:
fact = steps.build_fact(spark)

In [2]:
fact.select(
    "order_id", "courier_sk", "region_sk", "aoi_sk", "accept_date_key",
    "delivery_date_key", "accept_hour_key", "delivery_minutes",
).show(5, truncate=False)
print("Số dòng fact:", fact.count())
print("order_id duy nhất:", fact.select("order_id").distinct().count())

+--------+--------------------+---------+------+---------------+-----------------+---------------+----------------+
|order_id|courier_sk          |region_sk|aoi_sk|accept_date_key|delivery_date_key|accept_hour_key|delivery_minutes|
+--------+--------------------+---------+------+---------------+-----------------+---------------+----------------+
|2071340 |-5931354023872468343|1        |14    |20220618       |20220618         |8              |501.0           |
|3767291 |-5931354023872468343|1        |14    |20220814       |20220814         |8              |391.0           |
|2184812 |-5931354023872468343|1        |14    |20220901       |20220901         |9              |367.0           |
|4048066 |-6655371896403481998|1        |28    |20220825       |20220825         |13             |267.0           |
|2309133 |-306575199876071777 |1        |29    |20220709       |20220709         |8              |119.0           |
+--------+--------------------+---------+------+---------------+--------

### 2. Xem tập dùng cho Spark

Giữ đơn có GPS lúc nhận hợp lệ, khoảng cách tới đích tối đa 50 km và thời
lượng tối đa 24 giờ. Cell gọi đúng bộ lọc của Spark LR/GBT.

In [ ]:
cohort = steps.spark_baseline_cohort(fact)
print("Số đơn đủ điều kiện vào tập học Spark", cohort.count())
cohort.select("order_id", "order_date", "delivery_minutes", "distance_km").show(5, truncate=False)

### 3. Chia theo ngày và fit hai baseline

Train gồm ngày nhận trước 01/10/2022. Test bắt đầu từ ngày đó. Hai mô hình
học `log1p(delivery_minutes)`. Khi chấm thì đổi bằng `expm1` về phút và chặn dưới ở 0.

`steps.train_spark_baseline()` ghi `eta_predictions_gold`. Prefect gọi
`build_fact_and_train()` để dựng fact và fit liền nhau.

In [1]:
result = steps.train_spark_baseline(fact, train_end="2022-10-01")
result

{'Baseline: trung vị': {'mae': 127.59, 'rmse': 169.45, 'r2': -0.0494},
 'Baseline: trung bình': {'mae': 131.0, 'rmse': 165.42, 'r2': -0.0001},
 'Linear Regression': {'mae': 109.88, 'rmse': 168.64, 'r2': -0.0394},
 'GBT': {'mae': 92.29, 'rmse': 136.19, 'r2': 0.322},
 'train_rows': 165222,
 'test_rows': 35778,
 'late_threshold_min': 467.0}